![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de popularidad en canción

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de popularidad en canción".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la competencia de Kaggle cuyo link estará disponible en la sección del Coursera del proyecto.

## Datos para la predicción de popularidad en cancion

En este proyecto se usará el conjunto de datos de datos de popularidad en canciones, donde cada observación representa una canción y se tienen variables como: duración de la canción, acusticidad y tempo, entre otras. El objetivo es predecir qué tan popular es la canción. Para más detalles puede visitar el siguiente enlace: [datos](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

## Ejemplo predicción conjunto de test para envío a Kaggle

En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv', index_col=0)

In [4]:
# Visualización datos de entrenamiento
dataTraining.head()

,Unnamed: 0,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,0,7hUhmkALyQ8SX9mJs5XI3D,Love and Rockets,Love and Rockets,Motorcycle,211533,False,0.305,0.8490,9,...,1,0.0549,0.000058,0.056700,0.4640,0.3200,141.793,4,goth,22
1,1,5x59U89ZnjZXuNAAlc8X1u,Filippa Giordano,Filippa Giordano,"Addio del passato - From ""La traviata""",196000,False,0.287,0.1900,7,...,0,0.0370,0.930000,0.000356,0.0834,0.1330,83.685,4,opera,22
2,2,70Vng5jLzoJLmeLu3ayBQq,Susumu Yokota,Symbol,Purple Rose Minuet,216506,False,0.583,0.5090,1,...,1,0.0362,0.777000,0.202000,0.1150,0.5440,90.459,3,idm,37
3,3,1cRfzLJapgtwJ61xszs37b,Franz Liszt;YUNDI,Relajación y siestas,"Liebeslied (Widmung), S. 566",218346,False,0.163,0.0368,8,...,1,0.0472,0.991000,0.899000,0.1070,0.0387,69.442,3,classical,0
4,4,47d5lYjbiMy0EdMRV8lRou,Scooter,Scooter Forever,The Darkside,173160,False,0.647,0.9210,2,...,1,0.1850,0.000939,0.371000,0.1310,0.1710,137.981,4,techno,27


In [5]:
# Visualización datos de test
dataTesting.head()

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6KwkVtXm8OUp2XffN5k7lY,Hillsong Worship,No Other Name,No Other Name,440247,False,0.369,0.598,7,-6.984,1,0.0304,0.00511,0.000000,0.176,0.0466,148.014,4,world-music
1,2dp5I5MJ8bQQHDoFaNRFtX,Internal Rot,Grieving Birth,Failed Organum,93933,False,0.171,0.997,7,-3.586,1,0.1180,0.00521,0.801000,0.420,0.0294,122.223,4,grindcore
2,5avw06usmFkFrPjX8NxC40,Zhoobin Askarieh;Ali Sasha,Noise A Noise 20.4-1,"Save the Trees, Pt. 1",213578,False,0.173,0.803,9,-10.071,0,0.1440,0.61300,0.001910,0.195,0.0887,75.564,3,iranian
3,75hT0hvlESnDJstem0JgyR,Bryan Adams,All I Want For Christmas Is You,Merry Christmas,151387,False,0.683,0.511,6,-5.598,1,0.0279,0.40600,0.000197,0.111,0.5980,109.991,3,rock
4,4bY2oZGA5Br3pTE1Jd1IfY,Nogizaka46,バレッタ TypeD,月の大きさ,236293,False,0.555,0.941,9,-3.294,0,0.0481,0.48400,0.000000,0.266,0.8130,92.487,4,j-idol


In [6]:
dataTraining["popularity"].min()

np.int64(0)

In [7]:
len(dataTesting.columns)
len(dataTraining.columns)

21

In [8]:
dataTraining.info()

<class 'pandas.DataFrame'>
RangeIndex: 79800 entries, 0 to 79799
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        79800 non-null  int64  
 1   track_id          79800 non-null  str    
 2   artists           79800 non-null  str    
 3   album_name        79800 non-null  str    
 4   track_name        79800 non-null  str    
 5   duration_ms       79800 non-null  int64  
 6   explicit          79800 non-null  bool   
 7   danceability      79800 non-null  float64
 8   energy            79800 non-null  float64
 9   key               79800 non-null  int64  
 10  loudness          79800 non-null  float64
 11  mode              79800 non-null  int64  
 12  speechiness       79800 non-null  float64
 13  acousticness      79800 non-null  float64
 14  instrumentalness  79800 non-null  float64
 15  liveness          79800 non-null  float64
 16  valence           79800 non-null  float64
 17  temp

In [9]:
dataTraining.isna().sum()

Unnamed: 0          0
track_id            0
artists             0
album_name          0
track_name          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
popularity          0
dtype: int64

In [10]:
(dataTraining=="").sum()

Unnamed: 0          0
track_id            0
artists             0
album_name          0
track_name          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
popularity          0
dtype: int64

In [11]:
dataTraining[dataTraining.isna().any(axis=1)]

,Unnamed: 0,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity


In [12]:
dataTraining.duplicated().sum()

np.int64(0)

In [13]:
(dataTraining.nunique() / len(dataTraining) * 100).sort_values(ascending=False)

Unnamed: 0          100.000000
track_id             83.609023
track_name           69.883459
duration_ms          51.017544
album_name           46.760652
tempo                46.731830
artists              32.299499
loudness             22.007519
instrumentalness      6.581454
acousticness          6.085213
energy                2.421053
valence               2.176692
liveness              2.137845
speechiness           1.822055
danceability          1.403509
track_genre           0.142857
popularity            0.126566
key                   0.015038
time_signature        0.006266
mode                  0.002506
explicit              0.002506
dtype: float64

In [15]:
dataTransformada = dataTraining.drop(columns=['Unnamed: 0', 'track_id','track_name','artists', 'album_name'], inplace=False)

In [16]:
dataTransformada = pd.get_dummies(dataTransformada, columns=['track_genre'])

In [17]:
dataTransformada.columns
dataTransformada.columns[dataTransformada.columns.str.startswith('track_genre')]

Index(['track_genre_acoustic', 'track_genre_afrobeat', 'track_genre_alt-rock',
       'track_genre_alternative', 'track_genre_ambient', 'track_genre_anime',
       'track_genre_black-metal', 'track_genre_bluegrass', 'track_genre_blues',
       'track_genre_brazil',
       ...
       'track_genre_spanish', 'track_genre_study', 'track_genre_swedish',
       'track_genre_synth-pop', 'track_genre_tango', 'track_genre_techno',
       'track_genre_trance', 'track_genre_trip-hop', 'track_genre_turkish',
       'track_genre_world-music'],
      dtype='str', length=114)

In [18]:
XTotal = dataTransformada.drop(columns=['popularity'])
yTotal = dataTransformada['popularity']

In [19]:
XTrain, XTest, yTrain, yTest = train_test_split(XTotal, yTotal, test_size=0.33, random_state=17)

In [60]:
resultados_XGB = {"lr": [], "gamma": [], "colsample": [], "RMSE": [], "MAE": []}

learning_rates = np.arange(0.01, 0.26, 0.03)
gammas         = np.arange(0, 5.5, 1.0)
colsamples     = np.arange(0.3, 1.1, 0.1)

mejor_rmse   = float('inf')
mejor_params = {}
total        = len(learning_rates) * len(gammas) * len(colsamples)
iteracion    = 0

for lr in learning_rates:
    for g in gammas:
        for cs in colsamples:
            iteracion += 1

            xgb_ajustado = XGBRegressor(
                objective='reg:squarederror',
                random_state=17,
                learning_rate=lr,
                gamma=g,
                colsample_bytree=cs
            )
            respuesta = cross_val_score(xgb_ajustado, XTrain, yTrain,
                                        cv=5, scoring="neg_mean_squared_error")
            rmse = np.sqrt(-respuesta.mean())
            mae_cv = cross_val_score(xgb_ajustado, XTrain, yTrain,
                                      cv=5, scoring="neg_mean_absolute_error")
            mae = -mae_cv.mean()

            resultados_XGB["lr"].append(lr)
            resultados_XGB["gamma"].append(g)
            resultados_XGB["colsample"].append(cs)
            resultados_XGB["RMSE"].append(rmse)
            resultados_XGB["MAE"].append(mae)

            if rmse < mejor_rmse:
                mejor_rmse = rmse
                mejor_params = {"lr": lr, "gamma": g, "colsample": cs}

            print(f"Iteración {iteracion}/{total} | RMSE: {rmse:.2f}")

print(f"\nMejor RMSE: {mejor_rmse:.2f}")
print(f"Mejores parámetros: {mejor_params}")

Iteración 1/432 | RMSE: 21.44
Iteración 2/432 | RMSE: 21.39
Iteración 3/432 | RMSE: 21.36
Iteración 4/432 | RMSE: 21.36
Iteración 5/432 | RMSE: 21.35
Iteración 6/432 | RMSE: 21.36
Iteración 7/432 | RMSE: 21.33
Iteración 8/432 | RMSE: 21.32
Iteración 9/432 | RMSE: 21.44
Iteración 10/432 | RMSE: 21.39
Iteración 11/432 | RMSE: 21.36
Iteración 12/432 | RMSE: 21.36
Iteración 13/432 | RMSE: 21.35
Iteración 14/432 | RMSE: 21.36
Iteración 15/432 | RMSE: 21.33
Iteración 16/432 | RMSE: 21.32
Iteración 17/432 | RMSE: 21.44
Iteración 18/432 | RMSE: 21.39
Iteración 19/432 | RMSE: 21.36
Iteración 20/432 | RMSE: 21.36
Iteración 21/432 | RMSE: 21.35
Iteración 22/432 | RMSE: 21.36
Iteración 23/432 | RMSE: 21.33
Iteración 24/432 | RMSE: 21.32
Iteración 25/432 | RMSE: 21.44
Iteración 26/432 | RMSE: 21.39
Iteración 27/432 | RMSE: 21.36
Iteración 28/432 | RMSE: 21.36
Iteración 29/432 | RMSE: 21.35
Iteración 30/432 | RMSE: 21.36
Iteración 31/432 | RMSE: 21.33
Iteración 32/432 | RMSE: 21.32
Iteración 33/432 

In [62]:
resultados_XGB2 = {"n_est": [], "depth": [], "lr": [], "RMSE": []}
n_estimadores = [100, 200, 300, 500]
depths = [3, 5, 7, 9]
learning_rates = np.arange(0.23, 0.5, 0.03)

mejor_rmse = float('inf')
mejor_params = {}

for n in n_estimadores:
    for d in depths:
        for lr in learning_rates:
            xgb = XGBRegressor(
                objective='reg:squarederror',
                random_state=17,
                learning_rate=lr,
                gamma=3.0,
                colsample_bytree=1,
                n_estimators=n,
                max_depth=d
            )
            resp = cross_val_score(xgb, XTrain, yTrain, cv=5,
                                   scoring="neg_mean_squared_error")
            rmse = np.sqrt(-resp.mean())

            resultados_XGB2["n_est"].append(n)
            resultados_XGB2["depth"].append(d)
            resultados_XGB2["lr"].append(lr)
            resultados_XGB2["RMSE"].append(rmse)

            if rmse < mejor_rmse:
                mejor_rmse = rmse
                mejor_params = {"n_est": n, "depth": d, "lr": lr}

            print(f"n_est={n}, depth={d}, lr={lr:.2f} | RMSE: {rmse:.2f}")

print(f"\nMejor RMSE: {mejor_rmse:.2f}")
print(f"Mejores parámetros: {mejor_params}")

n_est=100, depth=3, lr=0.23 | RMSE: 19.30
n_est=100, depth=3, lr=0.26 | RMSE: 19.18
n_est=100, depth=3, lr=0.29 | RMSE: 19.12
n_est=100, depth=3, lr=0.32 | RMSE: 19.04
n_est=100, depth=3, lr=0.35 | RMSE: 18.99
n_est=100, depth=3, lr=0.38 | RMSE: 18.92
n_est=100, depth=3, lr=0.41 | RMSE: 18.88
n_est=100, depth=3, lr=0.44 | RMSE: 18.83
n_est=100, depth=3, lr=0.47 | RMSE: 18.78
n_est=100, depth=3, lr=0.50 | RMSE: 18.77
n_est=100, depth=5, lr=0.23 | RMSE: 18.73
n_est=100, depth=5, lr=0.26 | RMSE: 18.63
n_est=100, depth=5, lr=0.29 | RMSE: 18.53
n_est=100, depth=5, lr=0.32 | RMSE: 18.47
n_est=100, depth=5, lr=0.35 | RMSE: 18.45
n_est=100, depth=5, lr=0.38 | RMSE: 18.44
n_est=100, depth=5, lr=0.41 | RMSE: 18.39
n_est=100, depth=5, lr=0.44 | RMSE: 18.37
n_est=100, depth=5, lr=0.47 | RMSE: 18.30
n_est=100, depth=5, lr=0.50 | RMSE: 18.33
n_est=100, depth=7, lr=0.23 | RMSE: 18.28
n_est=100, depth=7, lr=0.26 | RMSE: 18.26
n_est=100, depth=7, lr=0.29 | RMSE: 18.17
n_est=100, depth=7, lr=0.32 | RMSE

In [63]:
resultados_XGB3 = {"n_est": [], "depth": [], "lr": [], "RMSE": []}

n_estimadores = [500, 700, 1000]
depths = [7, 9, 11, 13]
learning_rates = np.arange(0.05, 0.25, 0.03)

mejor_rmse = float('inf')
mejor_params = {}

for n in n_estimadores:
    for d in depths:
        for lr in learning_rates:
            xgb = XGBRegressor(
                objective='reg:squarederror',
                random_state=17,
                learning_rate=lr,
                gamma=3.0,
                colsample_bytree=1,
                n_estimators=n,
                max_depth=d
            )
            resp = cross_val_score(xgb, XTrain, yTrain, cv=5,
                                   scoring="neg_mean_squared_error")
            rmse = np.sqrt(-resp.mean())

            resultados_XGB3["n_est"].append(n)
            resultados_XGB3["depth"].append(d)
            resultados_XGB3["lr"].append(lr)
            resultados_XGB3["RMSE"].append(rmse)

            if rmse < mejor_rmse:
                mejor_rmse = rmse
                mejor_params = {"n_est": n, "depth": d, "lr": lr}

            print(f"n_est={n}, depth={d}, lr={lr:.2f} | RMSE: {rmse:.2f}")

print(f"\nMejor RMSE: {mejor_rmse:.2f}")
print(f"Mejores parámetros: {mejor_params}")

n_est=500, depth=7, lr=0.05 | RMSE: 18.22
n_est=500, depth=7, lr=0.08 | RMSE: 17.82
n_est=500, depth=7, lr=0.11 | RMSE: 17.58
n_est=500, depth=7, lr=0.14 | RMSE: 17.45
n_est=500, depth=7, lr=0.17 | RMSE: 17.35
n_est=500, depth=7, lr=0.20 | RMSE: 17.42
n_est=500, depth=7, lr=0.23 | RMSE: 17.41
n_est=500, depth=9, lr=0.05 | RMSE: 17.86
n_est=500, depth=9, lr=0.08 | RMSE: 17.55
n_est=500, depth=9, lr=0.11 | RMSE: 17.33
n_est=500, depth=9, lr=0.14 | RMSE: 17.32
n_est=500, depth=9, lr=0.17 | RMSE: 17.37
n_est=500, depth=9, lr=0.20 | RMSE: 17.40
n_est=500, depth=9, lr=0.23 | RMSE: 17.43
n_est=500, depth=11, lr=0.05 | RMSE: 17.69
n_est=500, depth=11, lr=0.08 | RMSE: 17.43
n_est=500, depth=11, lr=0.11 | RMSE: 17.32
n_est=500, depth=11, lr=0.14 | RMSE: 17.37
n_est=500, depth=11, lr=0.17 | RMSE: 17.46
n_est=500, depth=11, lr=0.20 | RMSE: 17.52
n_est=500, depth=11, lr=0.23 | RMSE: 17.62
n_est=500, depth=13, lr=0.05 | RMSE: 17.46
n_est=500, depth=13, lr=0.08 | RMSE: 17.35
n_est=500, depth=13, lr=0

In [22]:
xgb_final = XGBRegressor(
    objective='reg:squarederror',
    random_state=17,
    n_estimators=1000,
    max_depth=9,
    learning_rate=0.11,
    gamma=3.0,
    colsample_bytree=1.0
)

xgb_final.fit(XTrain, yTrain)
y_pred = xgb_final.predict(XTest)

# Métricas
rmse_final = np.sqrt(mean_squared_error(yTest, y_pred))
mae_final = mean_absolute_error(yTest, y_pred)

print(f"RMSE Final: {rmse_final:.2f}")
print(f"MAE  Final: {mae_final:.2f}")

RMSE Final: 16.93
MAE  Final: 11.36


In [24]:
import joblib
joblib.dump(xgb_final, "spotify.pkl", compress=3)
 
print("\nModelo guardado: spotify.pkl")


Modelo guardado: spotify.pkl


In [25]:
Test = dataTesting.drop(columns=[ 'track_id','track_name','artists', 'album_name'], inplace=False)
Test = pd.get_dummies(Test, columns=['track_genre'])

In [26]:
y_pred = pd.DataFrame(xgb_final.predict(Test), 
                       index=Test.index, 
                       columns=['Popularity'])

In [75]:
y_pred.head()

,Popularity
0,48.826206
1,12.978717
2,-4.689150
3,-0.612703
4,28.051371


In [41]:
pd.set_option('display.max_columns', None)
XTotal.head(3)

,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre_acoustic,track_genre_afrobeat,track_genre_alt-rock,track_genre_alternative,track_genre_ambient,track_genre_anime,track_genre_black-metal,track_genre_bluegrass,track_genre_blues,track_genre_brazil,track_genre_breakbeat,track_genre_british,track_genre_cantopop,track_genre_chicago-house,track_genre_children,track_genre_chill,track_genre_classical,track_genre_club,track_genre_comedy,track_genre_country,track_genre_dance,track_genre_dancehall,track_genre_death-metal,track_genre_deep-house,track_genre_detroit-techno,track_genre_disco,track_genre_disney,track_genre_drum-and-bass,track_genre_dub,track_genre_dubstep,track_genre_edm,track_genre_electro,track_genre_electronic,track_genre_emo,track_genre_folk,track_genre_forro,track_genre_french,track_genre_funk,track_genre_garage,track_genre_german,track_genre_gospel,track_genre_goth,track_genre_grindcore,track_genre_groove,track_genre_grunge,track_genre_guitar,track_genre_happy,track_genre_hard-rock,track_genre_hardcore,track_genre_hardstyle,track_genre_heavy-metal,track_genre_hip-hop,track_genre_honky-tonk,track_genre_house,track_genre_idm,track_genre_indian,track_genre_indie,track_genre_indie-pop,track_genre_industrial,track_genre_iranian,track_genre_j-dance,track_genre_j-idol,track_genre_j-pop,track_genre_j-rock,track_genre_jazz,track_genre_k-pop,track_genre_kids,track_genre_latin,track_genre_latino,track_genre_malay,track_genre_mandopop,track_genre_metal,track_genre_metalcore,track_genre_minimal-techno,track_genre_mpb,track_genre_new-age,track_genre_opera,track_genre_pagode,track_genre_party,track_genre_piano,track_genre_pop,track_genre_pop-film,track_genre_power-pop,track_genre_progressive-house,track_genre_psych-rock,track_genre_punk,track_genre_punk-rock,track_genre_r-n-b,track_genre_reggae,track_genre_reggaeton,track_genre_rock,track_genre_rock-n-roll,track_genre_rockabilly,track_genre_romance,track_genre_sad,track_genre_salsa,track_genre_samba,track_genre_sertanejo,track_genre_show-tunes,track_genre_singer-songwriter,track_genre_ska,track_genre_sleep,track_genre_songwriter,track_genre_soul,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,211533,False,0.305,0.849,9,-10.795,1,0.0549,0.000058,0.056700,0.4640,0.320,141.793,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,196000,False,0.287,0.190,7,-12.030,0,0.0370,0.930000,0.000356,0.0834,0.133,83.685,4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,216506,False,0.583,0.509,1,-9.6

In [33]:
yTotal.head()

0    22
1    22
2    37
3     0
4    27
Name: popularity, dtype: int64

In [54]:
df = pd.DataFrame([{
    "duration_ms": 211533,
    "explicit": 0,
    "danceability": 0.305,
    "energy": 0.849,
    "key": 9,
    "loudness": -10.795,
    "mode": 1,
    "speechiness": 0.0549,
    "acousticness": 0.000058,
    "instrumentalness": 0.0567,
    "liveness": 0.4640,
    "valence": 0.320,
    "tempo": 141.793,
    "time_signature": 4,
    "track_genre_goth": 1
}])

df = df.reindex(columns=XTrain.columns, fill_value=0)

pred = xgb_final.predict(df)
print(pred)

[30.140135]


In [76]:
y_ceros = y_pred.copy()
y_ceros['Popularity'] = y_ceros['Popularity'].clip(lower=0)

In [77]:
y_ceros.head()

,Popularity
0,48.826206
1,12.978717
2,0.000000
3,0.000000
4,28.051371


In [74]:
y_pred.min()

Popularity   -13.13924
dtype: float32

In [ ]:
# Predicción del conjunto de test - acá se genera un número aleatorio como ejemplo
np.random.seed(42)
y_pred = pd.DataFrame(np.random.rand(dataTesting.shape[0]) * 100, index=dataTesting.index, columns=['Popularity'])

In [73]:
# Guardar predicciones en formato exigido en la competencia de kaggle
y_pred.to_csv('XGB_1.csv', index_label='ID')
y_pred.head()

,Popularity
0,48.826206
1,12.978717
2,-4.689150
3,-0.612703
4,28.051371


In [78]:
# Guardar predicciones en formato exigido en la competencia de kaggle
y_ceros.to_csv('XGB_ceros.csv', index_label='ID')
y_ceros.head()

,Popularity
0,48.826206
1,12.978717
2,0.000000
3,0.000000
4,28.051371


## 

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mejor_rmse = float('inf')
mejor_peso = 0

for w in np.arange(0, 1.05, 0.05):
    rmses_fold = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(XTrain), 1):
        X_tr, X_val = XTrain.iloc[train_idx], XTrain.iloc[val_idx]
        y_tr, y_val = yTrain.iloc[train_idx], yTrain.iloc[val_idx]
        
        rf = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=17)
        rf.fit(X_tr, y_tr)
        
        xgb = XGBRegressor(objective='reg:squarederror', random_state=17,
                           n_estimators=1000, max_depth=9, learning_rate=0.11,
                           gamma=3.0, colsample_bytree=1.0)
        xgb.fit(X_tr, y_tr)
        
        pred = w * xgb.predict(X_val) + (1 - w) * rf.predict(X_val)
        rmse_fold = np.sqrt(mean_squared_error(y_val, pred))
        rmses_fold.append(rmse_fold)
        print(f"  XGB: {w:.2f} | RF: {1-w:.2f} | Fold {fold}/5 | RMSE: {rmse_fold:.2f}")
    
    rmse_mean = np.mean(rmses_fold)
    
    if rmse_mean < mejor_rmse:
        mejor_rmse = rmse_mean
        mejor_peso = w
    
    print(f"  >>> Promedio RMSE: {rmse_mean:.2f}")
    print("-" * 60)

print(f"\nMejor RMSE: {mejor_rmse:.2f}")
print(f"Mejor peso XGB: {mejor_peso:.2f} | RF: {1-mejor_peso:.2f}")

  XGB: 0.00 | RF: 1.00 | Fold 1/5 | RMSE: 20.64
  XGB: 0.00 | RF: 1.00 | Fold 2/5 | RMSE: 20.52
  XGB: 0.00 | RF: 1.00 | Fold 3/5 | RMSE: 20.52
  XGB: 0.00 | RF: 1.00 | Fold 4/5 | RMSE: 20.63
  XGB: 0.00 | RF: 1.00 | Fold 5/5 | RMSE: 20.59
  >>> Promedio RMSE: 20.58
------------------------------------------------------------
  XGB: 0.05 | RF: 0.95 | Fold 1/5 | RMSE: 20.28
  XGB: 0.05 | RF: 0.95 | Fold 2/5 | RMSE: 20.15
  XGB: 0.05 | RF: 0.95 | Fold 3/5 | RMSE: 20.16
  XGB: 0.05 | RF: 0.95 | Fold 4/5 | RMSE: 20.26
  XGB: 0.05 | RF: 0.95 | Fold 5/5 | RMSE: 20.22
  >>> Promedio RMSE: 20.21
------------------------------------------------------------
  XGB: 0.10 | RF: 0.90 | Fold 1/5 | RMSE: 19.93
  XGB: 0.10 | RF: 0.90 | Fold 2/5 | RMSE: 19.79
  XGB: 0.10 | RF: 0.90 | Fold 3/5 | RMSE: 19.81
  XGB: 0.10 | RF: 0.90 | Fold 4/5 | RMSE: 19.91
  XGB: 0.10 | RF: 0.90 | Fold 5/5 | RMSE: 19.87
  >>> Promedio RMSE: 19.86
------------------------------------------------------------
  XGB: 0.15 | RF

In [19]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

resultados_CAT = {"depth": [], "lr": [], "l2": [], "RMSE": []}

depths = [6, 8, 10, 12]
learning_rates = [0.05, 0.08, 0.11, 0.15, 0.2]
l2_regs = [1, 3, 5, 7, 9]

mejor_rmse = float('inf')
mejor_params = {}
total = len(depths) * len(learning_rates) * len(l2_regs)
iteracion = 0

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for d in depths:
    for lr in learning_rates:
        for l2 in l2_regs:
            iteracion += 1
            rmses_fold = []

            for fold, (train_idx, val_idx) in enumerate(kf.split(XTrain), 1):
                X_tr, X_val = XTrain.iloc[train_idx], XTrain.iloc[val_idx]
                y_tr, y_val = yTrain.iloc[train_idx], yTrain.iloc[val_idx]

                cat = CatBoostRegressor(
                    iterations=1000,
                    depth=d,
                    learning_rate=lr,
                    l2_leaf_reg=l2,
                    random_seed=17,
                    verbose=0
                )
                cat.fit(X_tr, y_tr)
                pred = cat.predict(X_val)
                rmse_fold = np.sqrt(mean_squared_error(y_val, pred))
                rmses_fold.append(rmse_fold)
                print(f"  Iter {iteracion}/{total} | depth={d}, lr={lr}, l2={l2} | Fold {fold}/5 | RMSE: {rmse_fold:.2f}")

            rmse_mean = np.mean(rmses_fold)

            resultados_CAT["depth"].append(d)
            resultados_CAT["lr"].append(lr)
            resultados_CAT["l2"].append(l2)

  Iter 1/100 | depth=6, lr=0.05, l2=1 | Fold 1/5 | RMSE: 18.32
  Iter 1/100 | depth=6, lr=0.05, l2=1 | Fold 2/5 | RMSE: 18.10
  Iter 1/100 | depth=6, lr=0.05, l2=1 | Fold 3/5 | RMSE: 18.20
  Iter 1/100 | depth=6, lr=0.05, l2=1 | Fold 4/5 | RMSE: 18.29
  Iter 1/100 | depth=6, lr=0.05, l2=1 | Fold 5/5 | RMSE: 18.26
  Iter 2/100 | depth=6, lr=0.05, l2=3 | Fold 1/5 | RMSE: 18.31
  Iter 2/100 | depth=6, lr=0.05, l2=3 | Fold 2/5 | RMSE: 18.14
  Iter 2/100 | depth=6, lr=0.05, l2=3 | Fold 3/5 | RMSE: 18.23
  Iter 2/100 | depth=6, lr=0.05, l2=3 | Fold 4/5 | RMSE: 18.31
  Iter 2/100 | depth=6, lr=0.05, l2=3 | Fold 5/5 | RMSE: 18.28
  Iter 3/100 | depth=6, lr=0.05, l2=5 | Fold 1/5 | RMSE: 18.34
  Iter 3/100 | depth=6, lr=0.05, l2=5 | Fold 2/5 | RMSE: 18.15
  Iter 3/100 | depth=6, lr=0.05, l2=5 | Fold 3/5 | RMSE: 18.26
  Iter 3/100 | depth=6, lr=0.05, l2=5 | Fold 4/5 | RMSE: 18.36
  Iter 3/100 | depth=6, lr=0.05, l2=5 | Fold 5/5 | RMSE: 18.29
  Iter 4/100 | depth=6, lr=0.05, l2=7 | Fold 1/5 | RMSE

KeyboardInterrupt: 

In [21]:
resultados_CAT2 = {"depth": [], "lr": [], "RMSE": []}

depths = [8, 10, 12, 14, 16, 18]
learning_rates = [0.19, 0.2, 0.21, 0.22]

mejor_rmse = float('inf')
mejor_params = {}
total = len(depths) * len(learning_rates)
iteracion = 0

kf = KFold(n_splits=5, shuffle=True, random_state=17)

for d in depths:
    for lr in learning_rates:
        iteracion += 1
        rmses_fold = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(XTrain), 1):
            X_tr, X_val = XTrain.iloc[train_idx], XTrain.iloc[val_idx]
            y_tr, y_val = yTrain.iloc[train_idx], yTrain.iloc[val_idx]

            cat = CatBoostRegressor(
                iterations=1000,
                depth=d,
                learning_rate=lr,
                l2_leaf_reg=3,
                random_seed=17,
                verbose=0
            )
            cat.fit(X_tr, y_tr)
            pred = cat.predict(X_val)
            rmse_fold = np.sqrt(mean_squared_error(y_val, pred))
            rmses_fold.append(rmse_fold)
            print(f"  Iter {iteracion}/{total} | depth={d}, lr={lr} | Fold {fold}/5 | RMSE: {rmse_fold:.2f}")

        rmse_mean = np.mean(rmses_fold)

        resultados_CAT2["depth"].append(d)
        resultados_CAT2["lr"].append(lr)
        resultados_CAT2["RMSE"].append(rmse_mean)

        if rmse_mean < mejor_rmse:
            mejor_rmse = rmse_mean
            mejor_params = {"depth": d, "lr": lr}

        print(f"  >>> Promedio RMSE: {rmse_mean:.2f}")
        print("-" * 70)

print(f"\nMejor RMSE: {mejor_rmse:.2f}")
print(f"Mejores parámetros: {mejor_params}")

  Iter 1/24 | depth=8, lr=0.19 | Fold 1/5 | RMSE: 17.01
  Iter 1/24 | depth=8, lr=0.19 | Fold 2/5 | RMSE: 16.97
  Iter 1/24 | depth=8, lr=0.19 | Fold 3/5 | RMSE: 17.40
  Iter 1/24 | depth=8, lr=0.19 | Fold 4/5 | RMSE: 17.22
  Iter 1/24 | depth=8, lr=0.19 | Fold 5/5 | RMSE: 17.17
  >>> Promedio RMSE: 17.15
----------------------------------------------------------------------
  Iter 2/24 | depth=8, lr=0.2 | Fold 1/5 | RMSE: 16.90
  Iter 2/24 | depth=8, lr=0.2 | Fold 2/5 | RMSE: 16.83
  Iter 2/24 | depth=8, lr=0.2 | Fold 3/5 | RMSE: 17.42
  Iter 2/24 | depth=8, lr=0.2 | Fold 4/5 | RMSE: 17.11
  Iter 2/24 | depth=8, lr=0.2 | Fold 5/5 | RMSE: 17.16
  >>> Promedio RMSE: 17.08
----------------------------------------------------------------------
  Iter 3/24 | depth=8, lr=0.21 | Fold 1/5 | RMSE: 16.95
  Iter 3/24 | depth=8, lr=0.21 | Fold 2/5 | RMSE: 16.90
  Iter 3/24 | depth=8, lr=0.21 | Fold 3/5 | RMSE: 17.34
  Iter 3/24 | depth=8, lr=0.21 | Fold 4/5 | RMSE: 17.19
  Iter 3/24 | depth=8, lr

CatBoostError: catboost/private/libs/options/oblivious_tree_options.cpp:128: Maximum tree depth is 16

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CARGA DE DATOS
# ============================================================
print("=" * 70)
print("CARGA DE DATOS")
print("=" * 70)

dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
print(f"Filas: {dataTraining.shape[0]} | Columnas: {dataTraining.shape[1]}")

# Muestra 10% para pruebas rápidas
data_sample, _ = train_test_split(dataTraining, train_size=0.1, random_state=17)
print(f"Muestra 10%: {data_sample.shape[0]} filas")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# FUNCIONES DE MODELOS (PARÁMETROS FIJOS)
# ============================================================
def get_xgb():
    return XGBRegressor(
        objective='reg:squarederror', random_state=17,
        n_estimators=1000, max_depth=9, learning_rate=0.11,
        gamma=3.0, colsample_bytree=1.0
    )

def get_cat():
    return CatBoostRegressor(
        iterations=1000, depth=14, learning_rate=0.19,
        l2_leaf_reg=3, random_seed=17, verbose=0
    )

def evaluar(X, y, modelo, nombre):
    scores = cross_val_score(modelo, X, y, cv=kf, scoring="neg_mean_squared_error")
    rmse = np.sqrt(-scores.mean())
    print(f"  {nombre}: RMSE = {rmse:.4f}")
    return rmse

# ============================================================
# MÉTODO 0: BASELINE (sin mejoras)
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 0: BASELINE")
print("=" * 70)

data_base = data_sample.drop(columns=['Unnamed: 0', 'track_id', 'track_name', 'artists', 'album_name'])
data_base = pd.get_dummies(data_base, columns=['track_genre'])
data_base.columns = data_base.columns.str.replace(r'[\[\]<]', '', regex=True)

X_base = data_base.drop(columns=['popularity'])
y_base = data_base['popularity']

rmse_xgb_base = evaluar(X_base, y_base, get_xgb(), "XGBoost baseline")
rmse_cat_base = evaluar(X_base, y_base, get_cat(), "CatBoost baseline")

# ============================================================
# MÉTODO 1: FEATURE ENGINEERING - Interacciones
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 1: FEATURE ENGINEERING - Interacciones")
print("=" * 70)

def agregar_interacciones(df):
    df = df.copy()
    df['energy_dance'] = df['energy'] * df['danceability']
    df['loud_energy'] = df['loudness'] * df['energy']
    df['tempo_dance'] = df['tempo'] * df['danceability']
    df['acoustic_energy'] = df['acousticness'] * df['energy']
    df['speech_dance'] = df['speechiness'] * df['danceability']
    df['valence_energy'] = df['valence'] * df['energy']
    df['live_acoustic'] = df['liveness'] * df['acousticness']
    df['loud_dance'] = df['loudness'] * df['danceability']
    return df

X_inter = agregar_interacciones(X_base)
rmse_xgb_inter = evaluar(X_inter, y_base, get_xgb(), "XGBoost + interacciones")
rmse_cat_inter = evaluar(X_inter, y_base, get_cat(), "CatBoost + interacciones")

# ============================================================
# MÉTODO 2: FEATURE ENGINEERING - Transformaciones
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 2: FEATURE ENGINEERING - Transformaciones")
print("=" * 70)

def agregar_transformaciones(df):
    df = df.copy()
    df['duration_min'] = df['duration_ms'] / 60000
    df['loudness_sq'] = df['loudness'] ** 2
    df['loudness_abs'] = np.abs(df['loudness'])
    df['tempo_sq'] = df['tempo'] ** 2
    df['energy_sq'] = df['energy'] ** 2
    df['dance_sq'] = df['danceability'] ** 2
    df['log_duration'] = np.log1p(df['duration_ms'])
    df['sqrt_tempo'] = np.sqrt(df['tempo'])
    return df

X_trans = agregar_transformaciones(X_base)
rmse_xgb_trans = evaluar(X_trans, y_base, get_xgb(), "XGBoost + transformaciones")
rmse_cat_trans = evaluar(X_trans, y_base, get_cat(), "CatBoost + transformaciones")

# ============================================================
# MÉTODO 3: TODAS LAS FEATURES JUNTAS
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 3: INTERACCIONES + TRANSFORMACIONES")
print("=" * 70)

X_all_fe = agregar_interacciones(X_base)
X_all_fe = agregar_transformaciones(X_all_fe)
rmse_xgb_all = evaluar(X_all_fe, y_base, get_xgb(), "XGBoost + todas FE")
rmse_cat_all = evaluar(X_all_fe, y_base, get_cat(), "CatBoost + todas FE")

# ============================================================
# MÉTODO 4: TARGET ENCODING DE ARTISTAS
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 4: TARGET ENCODING DE ARTISTAS")
print("=" * 70)

data_te = data_sample.drop(columns=['Unnamed: 0', 'track_id', 'track_name', 'album_name'])

# Target encoding con KFold para evitar data leakage
data_te['artist_encoded'] = 0.0
media_global = data_te['popularity'].mean()

for train_idx, val_idx in kf.split(data_te):
    media_artista = data_te.iloc[train_idx].groupby('artists')['popularity'].mean()
    data_te.iloc[val_idx, data_te.columns.get_loc('artist_encoded')] = \
        data_te.iloc[val_idx]['artists'].map(media_artista).fillna(media_global).values

# También encoding de cantidad de canciones por artista
conteo_artista = data_sample['artists'].value_counts()
data_te['artist_count'] = data_te['artists'].map(conteo_artista).fillna(1)

data_te = data_te.drop(columns=['artists'])
data_te = pd.get_dummies(data_te, columns=['track_genre'])
data_te.columns = data_te.columns.str.replace(r'[\[\]<]', '', regex=True)

X_te = data_te.drop(columns=['popularity'])
y_te = data_te['popularity']

rmse_xgb_te = evaluar(X_te, y_te, get_xgb(), "XGBoost + target encoding artista")
rmse_cat_te = evaluar(X_te, y_te, get_cat(), "CatBoost + target encoding artista")

# ============================================================
# MÉTODO 5: TARGET ENCODING + FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 5: TARGET ENCODING + FEATURE ENGINEERING")
print("=" * 70)

X_te_fe = agregar_interacciones(X_te)
X_te_fe = agregar_transformaciones(X_te_fe)

rmse_xgb_te_fe = evaluar(X_te_fe, y_te, get_xgb(), "XGBoost + TE + FE")
rmse_cat_te_fe = evaluar(X_te_fe, y_te, get_cat(), "CatBoost + TE + FE")

# ============================================================
# MÉTODO 6: TARGET ENCODING DE ÁLBUM
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 6: TARGET ENCODING ARTISTA + ÁLBUM")
print("=" * 70)

data_te2 = data_sample.drop(columns=['Unnamed: 0', 'track_id', 'track_name'])

# Target encoding artista
data_te2['artist_encoded'] = 0.0
for train_idx, val_idx in kf.split(data_te2):
    media_art = data_te2.iloc[train_idx].groupby('artists')['popularity'].mean()
    data_te2.iloc[val_idx, data_te2.columns.get_loc('artist_encoded')] = \
        data_te2.iloc[val_idx]['artists'].map(media_art).fillna(media_global).values

# Target encoding álbum
data_te2['album_encoded'] = 0.0
for train_idx, val_idx in kf.split(data_te2):
    media_alb = data_te2.iloc[train_idx].groupby('album_name')['popularity'].mean()
    data_te2.iloc[val_idx, data_te2.columns.get_loc('album_encoded')] = \
        data_te2.iloc[val_idx]['album_name'].map(media_alb).fillna(media_global).values

# Conteos
data_te2['artist_count'] = data_te2['artists'].map(conteo_artista).fillna(1)
conteo_album = data_sample['album_name'].value_counts()
data_te2['album_count'] = data_te2['album_name'].map(conteo_album).fillna(1)

data_te2 = data_te2.drop(columns=['artists', 'album_name'])
data_te2 = pd.get_dummies(data_te2, columns=['track_genre'])
data_te2.columns = data_te2.columns.str.replace(r'[\[\]<]', '', regex=True)

X_te2 = data_te2.drop(columns=['popularity'])
y_te2 = data_te2['popularity']

rmse_xgb_te2 = evaluar(X_te2, y_te2, get_xgb(), "XGBoost + TE artista+álbum")
rmse_cat_te2 = evaluar(X_te2, y_te2, get_cat(), "CatBoost + TE artista+álbum")

# Con FE
X_te2_fe = agregar_interacciones(X_te2)
X_te2_fe = agregar_transformaciones(X_te2_fe)
rmse_xgb_te2_fe = evaluar(X_te2_fe, y_te2, get_xgb(), "XGBoost + TE art+alb + FE")
rmse_cat_te2_fe = evaluar(X_te2_fe, y_te2, get_cat(), "CatBoost + TE art+alb + FE")

# ============================================================
# MÉTODO 7: ELIMINAR OUTLIERS
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 7: ELIMINAR OUTLIERS")
print("=" * 70)

q01 = data_sample['popularity'].quantile(0.01)
q99 = data_sample['popularity'].quantile(0.99)
data_no_out = data_sample[(data_sample['popularity'] >= q01) & (data_sample['popularity'] <= q99)]

data_no_out2 = data_no_out.drop(columns=['Unnamed: 0', 'track_id', 'track_name', 'artists', 'album_name'])
data_no_out2 = pd.get_dummies(data_no_out2, columns=['track_genre'])
data_no_out2.columns = data_no_out2.columns.str.replace(r'[\[\]<]', '', regex=True)

X_no_out = data_no_out2.drop(columns=['popularity'])
y_no_out = data_no_out2['popularity']

print(f"  Filas originales: {data_sample.shape[0]} | Sin outliers: {data_no_out.shape[0]}")
rmse_xgb_out = evaluar(X_no_out, y_no_out, get_xgb(), "XGBoost sin outliers")
rmse_cat_out = evaluar(X_no_out, y_no_out, get_cat(), "CatBoost sin outliers")

# ============================================================
# MÉTODO 8: TRANSFORMAR TARGET (Log)
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 8: TRANSFORMAR TARGET (log1p)")
print("=" * 70)

y_log = np.log1p(y_base)

# Evaluar manualmente porque necesitamos invertir la transformación
rmses_xgb_log = []
rmses_cat_log = []

for train_idx, val_idx in kf.split(X_base):
    X_tr, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_tr, y_val = y_log.iloc[train_idx], y_base.iloc[val_idx]  # y_val en escala original

    xgb = get_xgb()
    xgb.fit(X_tr, y_tr)
    pred_xgb = np.expm1(xgb.predict(X_val))  # Invertir log
    rmses_xgb_log.append(np.sqrt(mean_squared_error(y_val, pred_xgb)))

    cat = get_cat()
    cat.fit(X_tr, y_tr)
    pred_cat = np.expm1(cat.predict(X_val))
    rmses_cat_log.append(np.sqrt(mean_squared_error(y_val, pred_cat)))

print(f"  XGBoost + log target: RMSE = {np.mean(rmses_xgb_log):.4f}")
print(f"  CatBoost + log target: RMSE = {np.mean(rmses_cat_log):.4f}")

# ============================================================
# MÉTODO 9: TRANSFORMAR TARGET (sqrt)
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 9: TRANSFORMAR TARGET (sqrt)")
print("=" * 70)

y_sqrt = np.sqrt(np.clip(y_base, 0, None))

rmses_xgb_sqrt = []
rmses_cat_sqrt = []

for train_idx, val_idx in kf.split(X_base):
    X_tr, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_tr, y_val = y_sqrt.iloc[train_idx], y_base.iloc[val_idx]

    xgb = get_xgb()
    xgb.fit(X_tr, y_tr)
    pred_xgb = (xgb.predict(X_val)) ** 2  # Invertir sqrt
    rmses_xgb_sqrt.append(np.sqrt(mean_squared_error(y_val, pred_xgb)))

    cat = get_cat()
    cat.fit(X_tr, y_tr)
    pred_cat = (cat.predict(X_val)) ** 2
    rmses_cat_sqrt.append(np.sqrt(mean_squared_error(y_val, pred_cat)))

print(f"  XGBoost + sqrt target: RMSE = {np.mean(rmses_xgb_sqrt):.4f}")
print(f"  CatBoost + sqrt target: RMSE = {np.mean(rmses_cat_sqrt):.4f}")

# ============================================================
# MÉTODO 10: STACKING XGB + CAT
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 10: STACKING XGB + CAT")
print("=" * 70)

stacking = StackingRegressor(
    estimators=[('xgb', get_xgb()), ('cat', get_cat())],
    final_estimator=Ridge(random_state=17),
    cv=5
)
rmse_stack = evaluar(X_base, y_base, stacking, "Stacking XGB+CAT")

# ============================================================
# MÉTODO 11: ENSAMBLE PONDERADO XGB + CAT
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 11: ENSAMBLE PONDERADO XGB + CAT")
print("=" * 70)

mejor_rmse_ens = float('inf')
mejor_peso = 0

for w in np.arange(0, 1.05, 0.05):
    rmses_fold = []
    for train_idx, val_idx in kf.split(X_base):
        X_tr, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
        y_tr, y_val = y_base.iloc[train_idx], y_base.iloc[val_idx]

        xgb = get_xgb()
        xgb.fit(X_tr, y_tr)
        cat = get_cat()
        cat.fit(X_tr, y_tr)

        pred = w * xgb.predict(X_val) + (1 - w) * cat.predict(X_val)
        rmses_fold.append(np.sqrt(mean_squared_error(y_val, pred)))

    rmse_mean = np.mean(rmses_fold)
    if rmse_mean < mejor_rmse_ens:
        mejor_rmse_ens = rmse_mean
        mejor_peso = w
    print(f"  XGB: {w:.2f} | CAT: {1-w:.2f} | RMSE: {rmse_mean:.4f}")

print(f"\n  Mejor ensamble -> RMSE: {mejor_rmse_ens:.4f} | XGB={mejor_peso:.2f}, CAT={1-mejor_peso:.2f}")

# ============================================================
# MÉTODO 12: MEJOR COMBINACIÓN (TE + FE + Ensamble)
# ============================================================
print("\n" + "=" * 70)
print("MÉTODO 12: COMBINACIÓN FINAL (TE + FE + Ensamble)")
print("=" * 70)

# Usar el mejor dataset (con TE y FE)
mejor_rmse_final = float('inf')
mejor_peso_final = 0

for w in np.arange(0, 1.05, 0.05):
    rmses_fold = []
    for train_idx, val_idx in kf.split(X_te2_fe):
        X_tr, X_val = X_te2_fe.iloc[train_idx], X_te2_fe.iloc[val_idx]
        y_tr, y_val = y_te2.iloc[train_idx], y_te2.iloc[val_idx]

        xgb = get_xgb()
        xgb.fit(X_tr, y_tr)
        cat = get_cat()
        cat.fit(X_tr, y_tr)

        pred = w * xgb.predict(X_val) + (1 - w) * cat.predict(X_val)
        rmses_fold.append(np.sqrt(mean_squared_error(y_val, pred)))

    rmse_mean = np.mean(rmses_fold)
    if rmse_mean < mejor_rmse_final:
        mejor_rmse_final = rmse_mean
        mejor_peso_final = w

print(f"  Mejor RMSE final: {mejor_rmse_final:.4f}")
print(f"  Pesos: XGB={mejor_peso_final:.2f} | CAT={1-mejor_peso_final:.2f}")

# ============================================================
# RESUMEN FINAL
# ============================================================
print("\n" + "=" * 70)
print("RESUMEN FINAL - TODOS LOS MÉTODOS")
print("=" * 70)

resumen = pd.DataFrame({
    'Método': [
        'Baseline XGB', 'Baseline CAT',
        'Interacciones XGB', 'Interacciones CAT',
        'Transformaciones XGB', 'Transformaciones CAT',
        'Todas FE XGB', 'Todas FE CAT',
        'Target Enc artista XGB', 'Target Enc artista CAT',
        'TE + FE XGB', 'TE + FE CAT',
        'TE art+alb XGB', 'TE art+alb CAT',
        'TE art+alb + FE XGB', 'TE art+alb + FE CAT',
        'Sin outliers XGB', 'Sin outliers CAT',
        'Log target XGB', 'Log target CAT',
        'Sqrt target XGB', 'Sqrt target CAT',
        'Stacking XGB+CAT',
        'Ensamble ponderado base',
        'Ensamble TE+FE (FINAL)',
    ],
    'RMSE': [
        rmse_xgb_base, rmse_cat_base,
        rmse_xgb_inter, rmse_cat_inter,
        rmse_xgb_trans, rmse_cat_trans,
        rmse_xgb_all, rmse_cat_all,
        rmse_xgb_te, rmse_cat_te,
        rmse_xgb_te_fe, rmse_cat_te_fe,
        rmse_xgb_te2, rmse_cat_te2,
        rmse_xgb_te2_fe, rmse_cat_te2_fe,
        rmse_xgb_out, rmse_cat_out,
        np.mean(rmses_xgb_log), np.mean(rmses_cat_log),
        np.mean(rmses_xgb_sqrt), np.mean(rmses_cat_sqrt),
        rmse_stack,
        mejor_rmse_ens,
        mejor_rmse_final,
    ]
}).sort_values('RMSE')

print(resumen.to_string(index=False))
print("\n  >>> El mejor método es: " + resumen.iloc[0]['Método'] + f" con RMSE: {resumen.iloc[0]['RMSE']:.4f}")

CARGA DE DATOS
Filas: 79800 | Columnas: 21
Muestra 10%: 7980 filas

MÉTODO 0: BASELINE
  XGBoost baseline: RMSE = 19.8712
